# Phase 1 — Does cross-timestep wallet-trajectory information improve transaction classification?

Empirical, single-question test. **No neural networks**, no GNN, no transformer. Just hand-crafted causal trajectory features fed to a Random Forest, compared head-to-head against RF on the raw per-tx features alone.

## Decision rule
- **`ΔF1 ≥ +0.02`** (≥ 2 F1 points lift): cross-timestep signal exists in this dataset. A neural Phase-2 design is worth pursuing.
- **`+0.005 ≤ ΔF1 < +0.02`**: marginal. Borderline — depends on whether more careful neural design can squeeze more out.
- **`ΔF1 < +0.005`**: no meaningful lift. The temporal-GNN-on-Elliptic direction is a dead end on this dataset; honest conclusion is "RF on per-tx features is the strongest baseline and trajectory information from the bipartite addr↔tx structure does not improve it."

## Causality contract
For target transaction `T` at time `t`, every engineered feature uses **only** wallet-trajectory data from `τ < t` (strict). Implementation: `np.searchsorted(wallet_timeline, t, side='left')` cuts off at the first index `≥ t`, so every prior we sum/average over has `τ < t`. We never use `T`'s own row (or any tx at exactly `t`) as a "prior" of `T`'s incident wallets.

Historical **labels** of prior txs are used as features — they are part of the deployment-time information set (a fraud team would have labelled past txs before the new tx arrives).

## Data and split
- `transactions_data/txs_features.csv` (203,769 × 184 → 182 per-tx features), `txs_classes.csv` (1=illicit, 2=licit, 3=unknown), `txs_edgelist.csv` (within-timestep, unused here).
- `actors_data/{AddrTx,TxAddr}_edgelist.txt` (bipartite addr↔tx — the only cross-timestep bridge).
- **`wallets_features.txt` is NOT loaded.** Each row is a lifetime aggregate (constant across timesteps), which means it incorporates post-`t` info; using it would leak.
- **Temporal split**: train `t ≤ 34`, test `t ≥ 35` (paper protocol).

## Local-only
Paths resolve relative to repo root via `Path.cwd()` walk-up; no Colab/Drive imports. Wall clock on a laptop: ~3 minutes (most of it is the trajectory-feature-engineering cell).


In [30]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve,
)

# Walk up from CWD to find the repo root (the dir that has actors_data/ and transactions_data/)
ROOT = Path.cwd()
while not (ROOT / "actors_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
WALLETS_DIR      = ROOT / "actors_data"
TRANSACTIONS_DIR = ROOT / "transactions_data"
assert WALLETS_DIR.exists() and TRANSACTIONS_DIR.exists(), \
    f"Could not find data dirs from {Path.cwd()} (looked up to {ROOT})"

print(f"repo root: {ROOT}")

TRAIN_END     = 34       # paper split: train t<=34, test t>=35
N_TIME_STEPS  = 49
RANDOM_SEED   = 175
np.random.seed(RANDOM_SEED)


repo root: /Users/jarayliu/Documents/GitHub/stat175-final


In [31]:
print("Loading transactions...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")

tx_classes_df["label"] = tx_classes_df["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
tx_feat_cols = [c for c in tx_features_df.columns if c not in ("txId", "Time step")]
F_tx = len(tx_feat_cols)

merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_raw = tx_features_df[tx_feat_cols].fillna(0).astype(np.float32).values

print(f"  N_tx={N_tx:,}  F_tx={F_tx}")
print(f"  labels: illicit={int((tx_label==1).sum()):,}  licit={int((tx_label==0).sum()):,}  "
      f"unknown={int((tx_label==-1).sum()):,}")

print("\nLoading actor bipartite edges (addr<->tx)...")
addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")   # input wallet -> tx
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")   # tx -> output wallet

# Build wallet idx space from edge endpoints (we don't need wallet labels, only their identities/timelines).
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) | set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a: i for i, a in enumerate(wallet_addrs)}
N_w = len(wallet_addr_to_idx)
print(f"  unique wallets in addr<->tx edges: {N_w:,}")


Loading transactions...
  N_tx=203,769  F_tx=182
  labels: illicit=4,545  licit=42,019  unknown=157,205

Loading actor bipartite edges (addr<->tx)...
  unique wallets in addr<->tx edges: 822,942


In [32]:
print("Building per-wallet timelines (sorted by tx time)...")

def edges_with_time(edge_df, addr_col, tx_col):
    df = edge_df[[addr_col, tx_col]].copy()
    df["w"]  = df[addr_col].map(wallet_addr_to_idx)
    df["tx"] = df[tx_col].map(tx_id_to_idx)
    df = df.dropna(subset=["w", "tx"])
    df["w"]  = df["w"].astype(np.int64)
    df["tx"] = df["tx"].astype(np.int64)
    df["t"]      = tx_time[df["tx"].values]
    df["tx_lab"] = tx_label[df["tx"].values]
    return df[["w","tx","t","tx_lab"]]

at = edges_with_time(addr_tx_df, "input_address",  "txId")
ta = edges_with_time(tx_addr_df, "output_address", "txId")

# Per-tx incident input / output wallets
incident_in  = defaultdict(list)
incident_out = defaultdict(list)
for tx, w in zip(at["tx"].values, at["w"].values):
    incident_in[int(tx)].append(int(w))
for tx, w in zip(ta["tx"].values, ta["w"].values):
    incident_out[int(tx)].append(int(w))

# Per-wallet sorted timeline. Each wallet -> three parallel arrays (t, tx_idx, tx_label).
all_edges = pd.concat([at, ta], ignore_index=True)
# A wallet can be both input and output of the same tx -> dedup (w, tx) pairs
all_edges = all_edges.drop_duplicates(subset=["w", "tx"]).sort_values(["w", "t"]).reset_index(drop=True)
# group offsets
g = all_edges.groupby("w", sort=False)
wallet_t_arr   = {}
wallet_tx_arr  = {}
wallet_lab_arr = {}
for w, sub in g:
    wallet_t_arr[int(w)]   = sub["t"].values.astype(np.int64)
    wallet_tx_arr[int(w)]  = sub["tx"].values.astype(np.int64)
    wallet_lab_arr[int(w)] = sub["tx_lab"].values.astype(np.int64)

print(f"  wallets with timelines: {len(wallet_t_arr):,}")
print(f"  total wallet-tx incidence pairs: {len(all_edges):,}")


Building per-wallet timelines (sorted by tx time)...
  wallets with timelines: 822,942
  total wallet-tx incidence pairs: 1,268,260


In [33]:
# =============================================================================
#  CAUSAL WALLET-TRAJECTORY FEATURES FOR TRANSACTION CLASSIFICATION
# =============================================================================
#
#  What the existing Elliptic++ tx features already contain (183 features):
#    - 94 local features: this tx's properties (timestep, in/out degree,
#      BTC value, fees, output volume, averages of inputs/outputs)
#    - 72 aggregate features: 1-hop WITHIN-TIMESTEP neighbor statistics
#      (max/min/std/corr of same-timestep neighbors' local features)
#    - 17 Elliptic++ augmented features: this tx's BTC in/out totals,
#      fees, size, addr_in/out counts
#
#  ALL existing features are snapshot-local: they see nothing across timesteps.
#
#  What this code computes (NEW — zero overlap with existing features):
#    Cross-timestep wallet history through the bipartite addr↔tx bridge.
#    For each transaction T at time t, we look up its incident wallets,
#    retrieve each wallet's causal history (τ < t only), and aggregate:
#
#    GROUP 1 — Label propagation (strongest expected signal):
#      Number/fraction of prior illicit txs per wallet, max across wallets,
#      recency to last illicit tx, decayed illicit score, pre-thresholded
#      wallet counts with any illicit history.
#
#    GROUP 2 — Second-hop illicit exposure:
#      For each wallet w, count co-wallets (other wallets in w's prior txs)
#      that themselves have illicit history. This is the "guilt by association
#      through time" signal — a clean wallet transacting only with dirty
#      wallets is suspicious even without direct illicit labels.
#
#    GROUP 3 — Wallet structural profiles:
#      Fan-in/fan-out asymmetry (mixing services vs. normal users),
#      number of unique transaction partners, wallet disposability
#      (fraction of incident wallets appearing for the first time).
#
#    GROUP 4 — Temporal activity patterns:
#      Inter-arrival time statistics, burstiness coefficient,
#      recent activity velocity, wallet age.
#
#    GROUP 5 — Cross-side composites and anomaly ratios:
#      Combinations of input-side and output-side signals,
#      T's BTC volume vs. historical wallet volumes.
#
#  Causality invariant: per_wallet_priors(w, t_query) uses STRICT τ < t_query.
#  A wallet's first appearance at exactly t_T returns None (no priors).
#
# =============================================================================
#
#  PREREQUISITES — these variables must already exist from the data-loading cell:
#    tx_X_raw      : [N_tx, F_tx] float32  (raw un-standardised tx features)
#    tx_time       : [N_tx] int64           (timestep per tx)
#    tx_label      : [N_tx] int64           (1=illicit, 0=licit, -1=unknown)
#    tx_feat_cols  : list[str]              (column names for tx features)
#    incident_in   : dict[int, list[int]]   (tx_idx -> list of input wallet indices)
#    incident_out  : dict[int, list[int]]   (tx_idx -> list of output wallet indices)
#    wallet_t_arr  : dict[int, np.ndarray]  (wallet -> sorted array of active timesteps)
#    wallet_tx_arr : dict[int, np.ndarray]  (wallet -> array of tx indices per timestep)
#    wallet_lab_arr: dict[int, np.ndarray]  (wallet -> array of tx labels per timestep)
#    N_tx, N_TIME_STEPS
#
# =============================================================================

import time
import numpy as np
import pandas as pd

print("=" * 70)
print("  Engineering causal wallet-trajectory features")
print("=" * 70)

# ── Named tx features we'll aggregate from wallet priors ──────────────
# These are the interpretable named columns at the end of the feature list.
# RF already uses them heavily; aggregating their history adds temporal depth.
agg_feat_names = ["total_BTC", "fees", "num_input_addresses", "num_output_addresses"]
agg_feat_idxs  = [tx_feat_cols.index(c) for c in agg_feat_names]
total_btc_idx  = tx_feat_cols.index("total_BTC")
F_agg = len(agg_feat_names)
print(f"  per-prior aggregation features: {agg_feat_names}")

# ── Caps and constants ────────────────────────────────────────────────
MAX_INCIDENT_PER_SIDE = 32       # cap incident wallets per side (heavy-tail safety)
MAX_CO_WALLETS        = 150      # cap 2-hop co-wallet set per wallet (compute budget)
RECENCY_SENTINEL      = N_TIME_STEPS * 2   # "never seen illicit" placeholder
DECAY_RATE            = 0.2      # exponential decay for illicit recency score

# ── Precompute wallet richness for top-K selection ────────────────────
wallet_event_count = {w: len(tarr) for w, tarr in wallet_t_arr.items()}

def pick_top_wallets(wlist, k=MAX_INCIDENT_PER_SIDE):
    """Select the k most-active wallets from wlist (by timeline length)."""
    if len(wlist) <= k:
        return list(wlist)
    cnts = np.array([wallet_event_count.get(w, 0) for w in wlist], dtype=np.int64)
    order = np.argsort(-cnts, kind="stable")
    return [wlist[i] for i in order[:k]]


# ── Precompute per-wallet illicit flag lookup (for 2-hop exposure) ────
# wallet_ever_illicit_by_t[w] = sorted array of timesteps at which w first
# accumulated illicit evidence. Used to quickly check "does co-wallet c have
# illicit history by time t?" without re-scanning all priors.
print("  precomputing per-wallet illicit evidence arrays...")
_t0 = time.time()
wallet_has_illicit_by = {}  # w -> min timestep at which illicit label first observed
for w, labs in wallet_lab_arr.items():
    illicit_mask = (labs == 1)
    if illicit_mask.any():
        tarr = wallet_t_arr[w]
        wallet_has_illicit_by[w] = int(tarr[illicit_mask].min())
print(f"    {len(wallet_has_illicit_by):,} wallets with any illicit history  ({time.time()-_t0:.1f}s)")


def per_wallet_priors(w, t_query):
    """Compute causal summary for wallet w using STRICT τ < t_query.

    Returns None if the wallet has no prior history (first appearance is at t_query).
    """
    tarr = wallet_t_arr.get(w)
    if tarr is None:
        return None
    cut = int(np.searchsorted(tarr, t_query, side="left"))   # strict < t_query
    if cut == 0:
        return None

    prior_t   = tarr[:cut]
    prior_lab = wallet_lab_arr[w][:cut]
    prior_tx  = wallet_tx_arr[w][:cut]

    # ── GROUP 1: Label propagation ────────────────────────────────────
    illicit_mask = (prior_lab == 1)
    licit_mask   = (prior_lab == 0)
    n_illicit    = int(illicit_mask.sum())
    n_licit      = int(licit_mask.sum())
    n_labelled   = n_illicit + n_licit

    last_illicit_t = int(prior_t[illicit_mask].max()) if n_illicit > 0 else -1

    # Decayed illicit score: recent illicit history weighted more
    if n_illicit > 0:
        illicit_times = prior_t[illicit_mask]
        decay_weights = np.exp(-DECAY_RATE * (t_query - illicit_times).astype(np.float64))
        decayed_illicit_score = float(decay_weights.sum())
    else:
        decayed_illicit_score = 0.0

    illicit_frac = n_illicit / max(n_labelled, 1)

    # ── GROUP 2: Second-hop illicit exposure ──────────────────────────
    # Collect co-wallets from w's prior txs (other wallets in the same txs)
    co_wallets = set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int, []):
            if co_w != w:
                co_wallets.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w:
                co_wallets.add(co_w)
        if len(co_wallets) >= MAX_CO_WALLETS:
            break   # cap for compute

    n_co_wallets = len(co_wallets)
    n_co_illicit = 0
    for co_w in co_wallets:
        first_ill = wallet_has_illicit_by.get(co_w)
        if first_ill is not None and first_ill < t_query:
            n_co_illicit += 1

    # ── GROUP 3: Wallet structural profile ────────────────────────────
    # Unique transaction partners (fan-in / fan-out asymmetry)
    unique_in_partners  = set()
    unique_out_partners = set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int, []):
            if co_w != w:
                unique_in_partners.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w:
                unique_out_partners.add(co_w)

    n_in_partners  = len(unique_in_partners)
    n_out_partners = len(unique_out_partners)
    fan_asymmetry  = n_out_partners / max(n_in_partners, 1)

    # ── GROUP 4: Temporal activity patterns ───────────────────────────
    n_prior = int(cut)
    age     = int(t_query - prior_t[0])
    recency = int(t_query - prior_t[-1])

    n_recent_5 = int(((t_query - prior_t) <= 5).sum())
    n_recent_1 = int(((t_query - prior_t) <= 1).sum())

    if n_prior >= 2:
        inter_arrival = np.diff(prior_t.astype(np.float64))
        iat_mean   = float(inter_arrival.mean())
        iat_std    = float(inter_arrival.std())
        burstiness = float((iat_std - iat_mean) / (iat_std + iat_mean + 1e-8))
    else:
        iat_mean   = 0.0
        iat_std    = 0.0
        burstiness = 0.0

    velocity = n_recent_5 / max(n_prior, 1)

    # ── Prior tx feature aggregations ─────────────────────────────────
    feat_vals = tx_X_raw[prior_tx][:, agg_feat_idxs]   # [k, F_agg]

    return {
        # Group 1: label propagation
        "n":                    n_prior,
        "n_illicit":            n_illicit,
        "n_licit":              n_licit,
        "illicit_frac":         illicit_frac,
        "last_illicit_t":       last_illicit_t,
        "decayed_illicit":      decayed_illicit_score,
        # Group 2: second-hop exposure
        "n_co_wallets":         n_co_wallets,
        "n_co_illicit":         n_co_illicit,
        "co_illicit_frac":      n_co_illicit / max(n_co_wallets, 1),
        # Group 3: structural profile
        "n_in_partners":        n_in_partners,
        "n_out_partners":       n_out_partners,
        "fan_asymmetry":        fan_asymmetry,
        # Group 4: temporal patterns
        "first_seen_t":         int(prior_t[0]),
        "last_seen_t":          int(prior_t[-1]),
        "age":                  age,
        "recency":              recency,
        "n_recent_5":           n_recent_5,
        "n_recent_1":           n_recent_1,
        "iat_mean":             iat_mean,
        "iat_std":              iat_std,
        "burstiness":           burstiness,
        "velocity":             velocity,
        # Prior tx feature stats
        "feat_means":           feat_vals.mean(axis=0),
        "feat_maxes":           feat_vals.max(axis=0),
    }


def aggregate_side(summaries, side, t_T):
    """Build per-side trajectory features from a list of per-wallet summaries.

    Each entry in `summaries` is either a dict (from per_wallet_priors) or
    None (wallet's first appearance is T itself — no priors available).
    """
    n_total   = len(summaries)
    valid     = [s for s in summaries if s is not None]
    n_w_prior = len(valid)

    out = {}
    p = side  # prefix shorthand

    # ── Wallet coverage ───────────────────────────────────────────────
    out[f"{p}_n_wallets"]              = n_total
    out[f"{p}_n_wallets_with_prior"]   = n_w_prior
    out[f"{p}_frac_first_appearance"]  = (n_total - n_w_prior) / max(n_total, 1)

    # ── Early return if no wallet has any prior history ────────────────
    if not valid:
        out[f"{p}_n_priors_sum"]                = 0
        out[f"{p}_n_priors_max"]                = 0
        # Group 1
        out[f"{p}_n_illicit_sum"]               = 0
        out[f"{p}_n_illicit_max"]               = 0
        out[f"{p}_n_licit_sum"]                 = 0
        out[f"{p}_n_wallets_with_illicit"]      = 0
        out[f"{p}_n_wallets_illicit_frac_gt0"]  = 0
        out[f"{p}_n_wallets_illicit_frac_gt50"] = 0
        out[f"{p}_illicit_frac_max"]            = 0.0
        out[f"{p}_illicit_frac_mean"]           = 0.0
        out[f"{p}_decayed_illicit_max"]         = 0.0
        out[f"{p}_decayed_illicit_sum"]         = 0.0
        out[f"{p}_recency_to_illicit_min"]      = RECENCY_SENTINEL
        # Group 2
        out[f"{p}_co_illicit_sum"]              = 0
        out[f"{p}_co_illicit_max"]              = 0
        out[f"{p}_co_illicit_frac_max"]         = 0.0
        out[f"{p}_co_illicit_frac_mean"]        = 0.0
        out[f"{p}_n_co_wallets_sum"]            = 0
        # Group 3
        out[f"{p}_fan_asymmetry_max"]           = 0.0
        out[f"{p}_fan_asymmetry_mean"]          = 0.0
        out[f"{p}_n_in_partners_max"]           = 0
        out[f"{p}_n_out_partners_max"]          = 0
        out[f"{p}_frac_single_use"]             = 0.0
        # Group 4
        out[f"{p}_age_max"]                     = 0
        out[f"{p}_age_mean"]                    = 0.0
        out[f"{p}_recency_min"]                 = RECENCY_SENTINEL
        out[f"{p}_n_recent_5_sum"]              = 0
        out[f"{p}_n_recent_5_max"]              = 0
        out[f"{p}_n_recent_1_sum"]              = 0
        out[f"{p}_velocity_max"]                = 0.0
        out[f"{p}_velocity_mean"]               = 0.0
        out[f"{p}_burstiness_max"]              = 0.0
        out[f"{p}_burstiness_mean"]             = 0.0
        out[f"{p}_iat_mean_min"]                = 0.0
        out[f"{p}_iat_std_max"]                 = 0.0
        # Prior tx feature aggregations
        for nm in agg_feat_names:
            out[f"{p}_prior_{nm}_mean_max"] = 0.0
            out[f"{p}_prior_{nm}_max_max"]  = 0.0
        return out

    # ── Vectorise wallet-level summaries ──────────────────────────────
    ns        = np.array([s["n"]              for s in valid], dtype=np.float64)
    nis       = np.array([s["n_illicit"]      for s in valid], dtype=np.float64)
    nls       = np.array([s["n_licit"]        for s in valid], dtype=np.float64)
    ill_frac  = np.array([s["illicit_frac"]   for s in valid], dtype=np.float64)
    dec_ill   = np.array([s["decayed_illicit"]for s in valid], dtype=np.float64)
    last_ill  = np.array([s["last_illicit_t"] for s in valid], dtype=np.int64)
    has_ill   = (last_ill >= 0)
    rec_ill   = np.where(has_ill, t_T - last_ill, RECENCY_SENTINEL).astype(np.float64)

    co_ill    = np.array([s["n_co_illicit"]     for s in valid], dtype=np.float64)
    co_n      = np.array([s["n_co_wallets"]     for s in valid], dtype=np.float64)
    co_frac   = np.array([s["co_illicit_frac"]  for s in valid], dtype=np.float64)

    fan_a     = np.array([s["fan_asymmetry"]    for s in valid], dtype=np.float64)
    n_inp     = np.array([s["n_in_partners"]    for s in valid], dtype=np.float64)
    n_outp    = np.array([s["n_out_partners"]   for s in valid], dtype=np.float64)

    ages      = np.array([s["age"]        for s in valid], dtype=np.float64)
    recs      = np.array([s["recency"]    for s in valid], dtype=np.float64)
    nr5       = np.array([s["n_recent_5"] for s in valid], dtype=np.float64)
    nr1       = np.array([s["n_recent_1"] for s in valid], dtype=np.float64)
    vel       = np.array([s["velocity"]   for s in valid], dtype=np.float64)
    burst     = np.array([s["burstiness"] for s in valid], dtype=np.float64)
    iat_m     = np.array([s["iat_mean"]   for s in valid], dtype=np.float64)
    iat_s     = np.array([s["iat_std"]    for s in valid], dtype=np.float64)

    feat_means = np.stack([s["feat_means"] for s in valid], axis=0)  # [k, F_agg]
    feat_maxes = np.stack([s["feat_maxes"] for s in valid], axis=0)

    # ── Aggregate across wallets ──────────────────────────────────────

    out[f"{p}_n_priors_sum"]                = int(ns.sum())
    out[f"{p}_n_priors_max"]                = int(ns.max())

    # Group 1: label propagation
    out[f"{p}_n_illicit_sum"]               = int(nis.sum())
    out[f"{p}_n_illicit_max"]               = int(nis.max())
    out[f"{p}_n_licit_sum"]                 = int(nls.sum())
    out[f"{p}_n_wallets_with_illicit"]      = int(has_ill.sum())
    out[f"{p}_n_wallets_illicit_frac_gt0"]  = int((ill_frac > 0.0).sum())
    out[f"{p}_n_wallets_illicit_frac_gt50"] = int((ill_frac > 0.5).sum())
    out[f"{p}_illicit_frac_max"]            = float(ill_frac.max())
    out[f"{p}_illicit_frac_mean"]           = float(ill_frac.mean())
    out[f"{p}_decayed_illicit_max"]         = float(dec_ill.max())
    out[f"{p}_decayed_illicit_sum"]         = float(dec_ill.sum())
    out[f"{p}_recency_to_illicit_min"]      = float(rec_ill.min())

    # Group 2: second-hop exposure
    out[f"{p}_co_illicit_sum"]              = int(co_ill.sum())
    out[f"{p}_co_illicit_max"]              = int(co_ill.max())
    out[f"{p}_co_illicit_frac_max"]         = float(co_frac.max())
    out[f"{p}_co_illicit_frac_mean"]        = float(co_frac.mean())
    out[f"{p}_n_co_wallets_sum"]            = int(co_n.sum())

    # Group 3: structural profile
    out[f"{p}_fan_asymmetry_max"]           = float(fan_a.max())
    out[f"{p}_fan_asymmetry_mean"]          = float(fan_a.mean())
    out[f"{p}_n_in_partners_max"]           = int(n_inp.max())
    out[f"{p}_n_out_partners_max"]          = int(n_outp.max())
    n_single_use = sum(1 for s in valid if s["n"] == 1)
    out[f"{p}_frac_single_use"]             = n_single_use / max(n_w_prior, 1)

    # Group 4: temporal patterns
    out[f"{p}_age_max"]                     = int(ages.max())
    out[f"{p}_age_mean"]                    = float(ages.mean())
    out[f"{p}_recency_min"]                 = int(recs.min())
    out[f"{p}_n_recent_5_sum"]              = int(nr5.sum())
    out[f"{p}_n_recent_5_max"]              = int(nr5.max())
    out[f"{p}_n_recent_1_sum"]              = int(nr1.sum())
    out[f"{p}_velocity_max"]                = float(vel.max())
    out[f"{p}_velocity_mean"]               = float(vel.mean())
    out[f"{p}_burstiness_max"]              = float(burst.max())
    out[f"{p}_burstiness_mean"]             = float(burst.mean())
    out[f"{p}_iat_mean_min"]                = float(iat_m.min()) if n_w_prior > 0 else 0.0
    out[f"{p}_iat_std_max"]                 = float(iat_s.max())

    # Prior tx feature aggregations
    for k, nm in enumerate(agg_feat_names):
        out[f"{p}_prior_{nm}_mean_max"] = float(feat_means[:, k].max())
        out[f"{p}_prior_{nm}_max_max"]  = float(feat_maxes[:, k].max())

    return out


# ── Main loop: compute trajectory features for every transaction ──────
print("\nComputing trajectory features for all transactions...")
t0 = time.time()
rows = []
PROGRESS = 25_000

for tx_idx in range(N_tx):
    t_T = int(tx_time[tx_idx])
    T_total_btc = float(tx_X_raw[tx_idx, total_btc_idx])

    in_w  = pick_top_wallets(incident_in.get(tx_idx, []))
    out_w = pick_top_wallets(incident_out.get(tx_idx, []))

    in_summ  = [per_wallet_priors(w, t_T) for w in in_w]
    out_summ = [per_wallet_priors(w, t_T) for w in out_w]

    row = {}
    row.update(aggregate_side(in_summ,  "in",  t_T))
    row.update(aggregate_side(out_summ, "out", t_T))

    # ── GROUP 5: Cross-side composites ────────────────────────────────
    row["both_sides_have_illicit"]      = int(
        row["in_n_wallets_with_illicit"] > 0 and row["out_n_wallets_with_illicit"] > 0
    )
    row["total_n_illicit_priors"]       = row["in_n_illicit_sum"] + row["out_n_illicit_sum"]
    row["total_n_wallets_with_illicit"] = (
        row["in_n_wallets_with_illicit"] + row["out_n_wallets_with_illicit"]
    )
    row["total_co_illicit"]             = row["in_co_illicit_sum"] + row["out_co_illicit_sum"]
    row["min_recency_to_illicit"]       = min(
        row["in_recency_to_illicit_min"], row["out_recency_to_illicit_min"]
    )
    row["max_illicit_frac_either_side"] = max(
        row["in_illicit_frac_max"], row["out_illicit_frac_max"]
    )
    row["max_decayed_illicit_either"]   = max(
        row["in_decayed_illicit_max"], row["out_decayed_illicit_max"]
    )
    row["max_co_illicit_frac_either"]   = max(
        row["in_co_illicit_frac_max"], row["out_co_illicit_frac_max"]
    )
    row["total_frac_first_appearance"]  = (
        (row["in_frac_first_appearance"] * max(row["in_n_wallets"], 1) +
         row["out_frac_first_appearance"] * max(row["out_n_wallets"], 1))
        / max(row["in_n_wallets"] + row["out_n_wallets"], 1)
    )

    # T-vs-history anomaly ratios (this tx's volume vs wallet history)
    max_prior_btc  = max(row.get("in_prior_total_BTC_max_max", 0),
                         row.get("out_prior_total_BTC_max_max", 0))
    mean_prior_btc = max(row.get("in_prior_total_BTC_mean_max", 0),
                         row.get("out_prior_total_BTC_mean_max", 0))
    row["T_btc_vs_max_prior"]  = T_total_btc / max(max_prior_btc, 1.0)
    row["T_btc_vs_mean_prior"] = T_total_btc / max(mean_prior_btc, 1.0)

    rows.append(row)

    if tx_idx > 0 and tx_idx % PROGRESS == 0:
        elapsed = time.time() - t0
        rate    = tx_idx / elapsed
        eta     = (N_tx - tx_idx) / rate
        print(f"  tx {tx_idx:>7,}/{N_tx:,}  ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

traj_df   = pd.DataFrame(rows)
traj_X    = traj_df.values.astype(np.float32)
traj_cols = list(traj_df.columns)
F_traj    = traj_X.shape[1]

elapsed = time.time() - t0
print(f"\n  Done: {F_traj} trajectory features × {N_tx:,} txs  ({elapsed:.1f}s)")

# ── Summary ───────────────────────────────────────────────────────────
print(f"\n  Feature groups:")
for prefix, label in [("in_", "input-side"), ("out_", "output-side")]:
    n = sum(1 for c in traj_cols if c.startswith(prefix))
    print(f"    {label:16s}: {n:3d} features")
n_cross = sum(1 for c in traj_cols if not c.startswith("in_") and not c.startswith("out_"))
print(f"    {'cross-side':16s}: {n_cross:3d} features")
print(f"    {'TOTAL':16s}: {F_traj:3d} features")

# ── Sanity check: one labelled example ────────────────────────────────
ex = 0
while tx_label[ex] == -1:
    ex += 1
print(f"\n  Example tx_idx={ex}  t={tx_time[ex]}  label={'illicit' if tx_label[ex]==1 else 'licit'}")
print(f"    in wallets: {len(incident_in.get(ex, []))}  out wallets: {len(incident_out.get(ex, []))}")
print(f"    total_n_illicit_priors       = {traj_df.loc[ex, 'total_n_illicit_priors']}")
print(f"    total_co_illicit             = {traj_df.loc[ex, 'total_co_illicit']}")
print(f"    min_recency_to_illicit       = {traj_df.loc[ex, 'min_recency_to_illicit']}")
print(f"    max_illicit_frac_either_side = {traj_df.loc[ex, 'max_illicit_frac_either_side']:.3f}")
print(f"    max_co_illicit_frac_either   = {traj_df.loc[ex, 'max_co_illicit_frac_either']:.3f}")
print(f"    max_decayed_illicit_either   = {traj_df.loc[ex, 'max_decayed_illicit_either']:.3f}")
print(f"    total_frac_first_appearance  = {traj_df.loc[ex, 'total_frac_first_appearance']:.3f}")
print(f"    T_btc_vs_max_prior           = {traj_df.loc[ex, 'T_btc_vs_max_prior']:.3f}")


  Engineering causal wallet-trajectory features
  per-prior aggregation features: ['total_BTC', 'fees', 'num_input_addresses', 'num_output_addresses']
  precomputing per-wallet illicit evidence arrays...
    14,266 wallets with any illicit history  (0.9s)

Computing trajectory features for all transactions...
  tx  25,000/203,769  (2s elapsed, ~18s remaining)
  tx  50,000/203,769  (6s elapsed, ~17s remaining)
  tx  75,000/203,769  (9s elapsed, ~16s remaining)
  tx 100,000/203,769  (14s elapsed, ~15s remaining)
  tx 125,000/203,769  (19s elapsed, ~12s remaining)
  tx 150,000/203,769  (21s elapsed, ~8s remaining)
  tx 175,000/203,769  (25s elapsed, ~4s remaining)
  tx 200,000/203,769  (28s elapsed, ~1s remaining)

  Done: 103 trajectory features × 203,769 txs  (32.2s)

  Feature groups:
    input-side      :  46 features
    output-side     :  46 features
    cross-side      :  11 features
    TOTAL           : 103 features

  Example tx_idx=3  t=1  label=licit
    in wallets: 1  out wal

In [34]:
print("=== Baseline: Random Forest on raw 182 per-tx features ===")
labelled   = (tx_label != -1)
train_mask = labelled & (tx_time <= TRAIN_END)
test_mask  = labelled & (tx_time >  TRAIN_END)

X_train_raw = tx_X_raw[train_mask]
X_test_raw  = tx_X_raw[test_mask]
y_train     = tx_label[train_mask]
y_test      = tx_label[test_mask]
test_t_arr  = tx_time[test_mask]

print(f"  train: n={X_train_raw.shape[0]:,}  illicit_rate={y_train.mean():.4f}")
print(f"  test:  n={X_test_raw.shape[0]:,}  illicit_rate={y_test.mean():.4f}")

t0 = time.time()
rf_raw = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                n_jobs=-1, random_state=RANDOM_SEED)
rf_raw.fit(X_train_raw, y_train)
print(f"  trained in {time.time()-t0:.1f}s")

y_pred_raw  = rf_raw.predict(X_test_raw)
y_proba_raw = rf_raw.predict_proba(X_test_raw)[:, 1]
f1_raw     = f1_score(y_test, y_pred_raw, pos_label=1, zero_division=0)
auc_raw    = roc_auc_score(y_test, y_proba_raw)
prauc_raw  = average_precision_score(y_test, y_proba_raw)
print(f"  F1={f1_raw:.4f}  AUC={auc_raw:.4f}  PR-AUC={prauc_raw:.4f}")
print(classification_report(y_test, y_pred_raw, target_names=["licit","illicit"], digits=4, zero_division=0))


=== Baseline: Random Forest on raw 182 per-tx features ===
  train: n=29,894  illicit_rate=0.1158
  test:  n=16,670  illicit_rate=0.0650
  trained in 1.8s
  F1=0.8044  AUC=0.9269  PR-AUC=0.7927
              precision    recall  f1-score   support

       licit     0.9781    0.9995    0.9887     15587
     illicit     0.9892    0.6777    0.8044      1083

    accuracy                         0.9786     16670
   macro avg     0.9837    0.8386    0.8965     16670
weighted avg     0.9788    0.9786    0.9767     16670



In [35]:
print("=== RF on raw 182 features + causal trajectory features ===")
X_train_aug = np.concatenate([X_train_raw, traj_X[train_mask]], axis=1)
X_test_aug  = np.concatenate([X_test_raw,  traj_X[test_mask]],  axis=1)
print(f"  augmented dim: {X_train_aug.shape[1]} = 182 + {F_traj} traj features")

t0 = time.time()
rf_aug = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                n_jobs=-1, random_state=RANDOM_SEED)
rf_aug.fit(X_train_aug, y_train)
print(f"  trained in {time.time()-t0:.1f}s")

y_pred_aug  = rf_aug.predict(X_test_aug)
y_proba_aug = rf_aug.predict_proba(X_test_aug)[:, 1]
f1_aug     = f1_score(y_test, y_pred_aug, pos_label=1, zero_division=0)
auc_aug    = roc_auc_score(y_test, y_proba_aug)
prauc_aug  = average_precision_score(y_test, y_proba_aug)
print(f"  F1={f1_aug:.4f}  AUC={auc_aug:.4f}  PR-AUC={prauc_aug:.4f}")
print(classification_report(y_test, y_pred_aug, target_names=["licit","illicit"], digits=4, zero_division=0))


=== RF on raw 182 features + causal trajectory features ===
  augmented dim: 285 = 182 + 103 traj features
  trained in 1.8s
  F1=0.8098  AUC=0.9333  PR-AUC=0.8097
              precision    recall  f1-score   support

       licit     0.9786    0.9994    0.9889     15587
     illicit     0.9880    0.6861    0.8098      1083

    accuracy                         0.9791     16670
   macro avg     0.9833    0.8427    0.8994     16670
weighted avg     0.9793    0.9791    0.9773     16670



In [36]:
print("=== Ablation: RF on trajectory features ONLY (no raw tx features) ===")
# Isolates how much of the augmented model's score is carried by the trajectory features
# vs. the per-tx features. If RF[traj only] is competitive with RF[raw], the trajectory
# signal is genuinely strong; if it's much weaker, the augmented gain is mostly from
# RF combining the two information sources rather than trajectory features alone.

X_train_traj_only = traj_X[train_mask]
X_test_traj_only  = traj_X[test_mask]

t0 = time.time()
rf_traj_only = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                      n_jobs=-1, random_state=RANDOM_SEED)
rf_traj_only.fit(X_train_traj_only, y_train)
print(f"  trained in {time.time()-t0:.1f}s")

y_pred_traj_only  = rf_traj_only.predict(X_test_traj_only)
y_proba_traj_only = rf_traj_only.predict_proba(X_test_traj_only)[:, 1]
f1_traj_only    = f1_score(y_test, y_pred_traj_only, pos_label=1, zero_division=0)
auc_traj_only   = roc_auc_score(y_test, y_proba_traj_only)
prauc_traj_only = average_precision_score(y_test, y_proba_traj_only)
print(f"  F1={f1_traj_only:.4f}  AUC={auc_traj_only:.4f}  PR-AUC={prauc_traj_only:.4f}")
print(classification_report(y_test, y_pred_traj_only,
                            target_names=["licit","illicit"], digits=4, zero_division=0))


=== Ablation: RF on trajectory features ONLY (no raw tx features) ===
  trained in 0.7s
  F1=0.5036  AUC=0.8493  PR-AUC=0.5600
              precision    recall  f1-score   support

       licit     0.9683    0.9561    0.9621     15587
     illicit     0.4648    0.5494    0.5036      1083

    accuracy                         0.9296     16670
   macro avg     0.7166    0.7527    0.7329     16670
weighted avg     0.9356    0.9296    0.9323     16670



In [37]:
print("=" * 60)
print("PHASE 1 VERDICT")
print("=" * 60)
print(f"  RF [raw 182 features]            F1={f1_raw:.4f}  AUC={auc_raw:.4f}  PR-AUC={prauc_raw:.4f}")
print(f"  RF [raw + {F_traj} traj features] F1={f1_aug:.4f}  AUC={auc_aug:.4f}  PR-AUC={prauc_aug:.4f}")
delta_f1    = f1_aug    - f1_raw
delta_auc   = auc_aug   - auc_raw
delta_prauc = prauc_aug - prauc_raw
print(f"  Δ                                F1={delta_f1:+.4f}  AUC={delta_auc:+.4f}  PR-AUC={delta_prauc:+.4f}")
print()
if delta_f1 >= 0.02:
    print("  ✅ Cross-timestep trajectory signal EXISTS (>= +0.02 F1).")
    print("     Phase 2 (end-to-end neural model that uses trajectory features as inputs) is justified.")
elif delta_f1 >= 0.005:
    print("  ⚠️  Marginal lift. Borderline. Probably not worth a fancy neural design.")
else:
    print("  ❌ No meaningful lift from cross-timestep trajectory information.")
    print("     Honest conclusion: RF on per-tx features is the strongest baseline on this dataset.")
    print("     The temporal-GNN-on-Elliptic direction is a dead end here.")

print("\n" + "=" * 60)
print("Per-timestep test F1 breakdown")
print("=" * 60)
print(f"{'t':>3}  {'n':>6}  {'illicit':>7}  {'RF[raw] F1':>10}  {'RF[aug] F1':>10}  {'Δ':>7}")
for t in range(TRAIN_END + 1, N_TIME_STEPS + 1):
    m = (test_t_arr == t)
    if not m.any():
        continue
    yt = y_test[m]
    if yt.sum() == 0:
        f1_r, f1_a, ds = float("nan"), float("nan"), "  N/A"
    else:
        f1_r = f1_score(yt, y_pred_raw[m], pos_label=1, zero_division=0)
        f1_a = f1_score(yt, y_pred_aug[m], pos_label=1, zero_division=0)
        ds = f"{f1_a - f1_r:+.3f}"
    print(f"{t:>3}  {int(m.sum()):>6}  {int(yt.sum()):>7}  {f1_r:>10.4f}  {f1_a:>10.4f}  {ds:>7}")

print("\n" + "=" * 60)
print("Top-30 features by RF[aug] importance (traj features marked)")
print("=" * 60)
all_feat_names = list(tx_feat_cols) + traj_cols
imp = rf_aug.feature_importances_
order = np.argsort(-imp)[:30]
for rank, i in enumerate(order, 1):
    tag = "  (TRAJ)" if i >= F_tx else ""
    print(f"  {rank:2d}.  {imp[i]:.4f}  {all_feat_names[i]}{tag}")

n_traj_in_top30 = sum(1 for i in order if i >= F_tx)
print(f"\n  trajectory features in top-30: {n_traj_in_top30} / 30")


PHASE 1 VERDICT
  RF [raw 182 features]            F1=0.8044  AUC=0.9269  PR-AUC=0.7927
  RF [raw + 103 traj features] F1=0.8098  AUC=0.9333  PR-AUC=0.8097
  Δ                                F1=+0.0054  AUC=+0.0064  PR-AUC=+0.0170

  ⚠️  Marginal lift. Borderline. Probably not worth a fancy neural design.

Per-timestep test F1 breakdown
  t       n  illicit  RF[raw] F1  RF[aug] F1        Δ
 35    1341      182      0.9805      0.9775   -0.003
 36    1708       33      0.8814      0.9000   +0.019
 37     498       40      0.7500      0.7692   +0.019
 38     756      111      0.9429      0.9479   +0.005
 39    1183       81      0.9128      0.9054   -0.007
 40    1211      112      0.7416      0.7486   +0.007
 41    1132      116      0.8900      0.9005   +0.011
 42    2154      239      0.8565      0.8599   +0.003
 43    1370       24      0.0000      0.0000   +0.000
 44    1591       24      0.0690      0.0667   -0.002
 45    1221        5      0.0000      0.0000   +0.000
 46     712  